In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.optimize import minimize, NonlinearConstraint

In [ ]:
# USER INPUTS
target_delta = 0.06      # Portfolio PINC
VAT = 0.19               # VAT for NR/Unit Calculation
VILC_GR = 0.0378         # VILC Annual Growth Rate

start_period = '2025-08'  # Optimization Starting Period
end_period = '2025-10'   # Optimizing Ending Period

In [ ]:
# READING IN ELASTICITY AND REFRENCE DATAFRAMES
own_to_own_elasticity_df = pd.read_csv(r'C:/Users/40107922/Downloads/r2/elasticity.csv')
reference_df = pd.read_csv(r'C:/Users/40107922/Downloads/r2/subset_reference_abi_sellin-vol_pl-ptc.csv')
own_to_competitor_elasticity_df = pd.read_csv(r'C:/Users/40107922/Downloads/r2/elasticity_competitor.csv')
competitor_reference_df = pd.read_csv(r'C:/Users/40107922/Downloads/r2/subset_reference_comp_sellout-vol_ptc.csv')
seg_mapping = pd.read_csv(r'C:/Users/40107922/Downloads/r2/segment_mapping.csv')

print(own_to_own_elasticity_df.shape, reference_df.shape, own_to_competitor_elasticity_df.shape, competitor_reference_df.shape)

In [ ]:
min_period = '2024-01'
max_period = '2026-12'

def parse_period(period_str):
    try:
        year_str, month_str = period_str.strip().split('-')
        year = int(year_str)
        month = int(month_str.lstrip('0') or '0')
        #year, month = map(int, period_str.strip().split('-'))
        if not (1<=month<=12):
            raise ValueError
        return year, month
    except Exception:
        raise ValueError(f"Invalid Period Format: '{period_str}'. Use 'YYYY-M'.")

def period_to_str(year, month):
    return f"{year}-{month:02d}"

def is_within_scope(year, month):
    min_year, min_month = parse_period(min_period)
    max_year, max_month = parse_period(max_period)
    return (year,month) >= (min_year, min_month) and (year,month) <= (max_year, max_month)


def get_valid_period(period_str, valid_periods):
    """Check if period is in the dataframe or fallback to previous year"""
    year, month = parse_period(period_str)
    
    if not is_within_scope(year, month):
        raise ValueError(f"Period '{period_str}' is out of allowed range ({min_period} to {max_period}).")

    if period_str in valid_periods:
        return period_str, False  # Use as-is, no fallback
    
    fallback_year = year - 1
    fallback_period = period_to_str(fallback_year, month)
    
    if not is_within_scope(fallback_year, month):
        raise ValueError(f"Neither '{period_str}' nor its fallback '{fallback_period}' are within allowed range.")
    
    if fallback_period in valid_periods:
        return fallback_period, True  # Use fallback
    else:
        raise ValueError(f"Period '{period_str}' and fallback '{fallback_period}' are not available in data.")


def filter_dataframe(df, start_period, end_period):
    valid_periods = set(df['year_month'])

    # Resolve periods
    start_resolved, start_flag = get_valid_period(start_period, valid_periods)
    end_resolved, end_flag = get_valid_period(end_period, valid_periods)

    # Parse to tuple for range filtering
    start_year, start_month = parse_period(start_resolved)
    end_year, end_month = parse_period(end_resolved)

    # Create a helper column for comparison
    df['ym_tuple'] = df['year_month'].apply(lambda x: parse_period(x))

    # Filter rows within range
    filtered_df = df[
        (df['ym_tuple'] >= (start_year, start_month)) & 
        (df['ym_tuple'] <= (end_year, end_month))
    ].copy()

    # Add the fallback flag
    filtered_df['used_year_ago'] = start_flag or end_flag

    # Drop helper column
    filtered_df.drop(columns='ym_tuple', inplace=True)
    print(filtered_df.shape)
    return filtered_df


try:
    reference_df_filt = filter_dataframe(reference_df, start_period, end_period)
except ValueError as e:
    print("Error:", e)


try:
    competitor_reference_df_filt = filter_dataframe(competitor_reference_df, start_period, end_period)
except ValueError as e:
    print("Error:", e)


reference_df_filt.loc[reference_df_filt['used_year_ago'], 'vilc'] = reference_df_filt['vilc'] * (1+ VILC_GR)

reference_df = reference_df_filt.drop(columns = {'used_year_ago'})
competitor_reference_df = competitor_reference_df_filt.drop(columns = {'used_year_ago'})

reference_df = reference_df.reset_index().drop(columns={'index'})
competitor_reference_df = competitor_reference_df.reset_index().drop(columns={'index'})

In [ ]:
reference_df.shape

In [ ]:
# PRE PROCESSING THE REFERENCE FILES

own_products_init = reference_df['sku'].unique()
competitor_products_init = competitor_reference_df['sku'].unique()

own_to_own_elasticity_df = own_to_own_elasticity_df[own_to_own_elasticity_df['target_sku'].isin(own_products_init)]
own_to_own_elasticity_df = own_to_own_elasticity_df[own_to_own_elasticity_df['other_sku'].isin(own_products_init)]

own_to_competitor_elasticity_df = own_to_competitor_elasticity_df[own_to_competitor_elasticity_df['target_sku'].isin(own_products_init)]
own_to_competitor_elasticity_df = own_to_competitor_elasticity_df[own_to_competitor_elasticity_df['other_sku'].isin(competitor_products_init)]

reference_df = reference_df[['year_month', 'sku', 'reference_volume', 'reference_price', 'markup', 'discount', 'excise', 'vilc', 'sellout_volume']]

# Extract unique SKUs from your elasticity dfs or existing reference dfs
own_skus = own_to_own_elasticity_df['target_sku'].unique()
own_skus.sort()
competitor_skus = own_to_competitor_elasticity_df['other_sku'].unique()
competitor_skus.sort()

# Extract all year_months from your data or define explicitly
all_months = reference_df['year_month'].unique()
# If your year_month is string, you might want to convert to datetime for sorting:
all_months_date = pd.to_datetime(all_months)
all_months_date = pd.Series(all_months_date).sort_values().unique()


# Build full MultiIndex for own products x all months
full_index_own = pd.MultiIndex.from_product(
    [all_months_date, own_skus],
    names=['year_month_date', 'sku']
)

# Convert reference_df_own['year_month'] to datetime for merging
reference_df['year_month_date'] = pd.to_datetime(reference_df['year_month'])

# Set index to year_month and sku for easier reindexing
reference_df = reference_df.set_index(['year_month_date', 'sku'])

# Reindex to full index
reference_df_own_padded = reference_df.reindex(full_index_own)

reference_df_own_padded['present'] = 'present'
reference_df_own_padded.loc[reference_df_own_padded['reference_volume'].isna(), 'present'] = 'missing'

# Fill missing volumes with zero
reference_df_own_padded['reference_volume'] = reference_df_own_padded['reference_volume'].fillna(0)
reference_df_own_padded['sellout_volume'] = reference_df_own_padded['sellout_volume'].fillna(0)

# Fill missing prices with the mean price per SKU (computed from existing data)
mean_prices = reference_df['reference_price'].mean()
reference_df_own_padded['reference_price'] =  reference_df_own_padded['reference_price'].fillna(mean_prices)

# Fill other numeric columns similarly or with zeros
for col in ['markup', 'discount', 'excise', 'vilc']:  # add columns as needed
    if col in reference_df_own_padded.columns:
        reference_df_own_padded[col] = reference_df_own_padded[col].fillna(0)

# Reset index if you want back to columns
reference_df_own_padded = reference_df_own_padded.reset_index()

year_month_mapping = reference_df.reset_index()
year_month_mapping = year_month_mapping[['year_month_date', 'year_month']].drop_duplicates()

reference_df_own_padded = pd.merge(reference_df_own_padded, year_month_mapping, how = 'left', on = 'year_month_date')

reference_df_own_padded = reference_df_own_padded.drop(columns = {'year_month_date', 'year_month_x'})
reference_df_own_padded.rename(columns = {'year_month_y': 'year_month'}, inplace = True)
reference_df_own_padded = pd.merge(reference_df_own_padded, seg_mapping, how = 'left', on = 'sku')
reference_df_padded_bound = reference_df_own_padded.copy()
reference_df_own_padded = reference_df_own_padded.drop(columns = {'present'})

# Extract all year_months from your data or define explicitly
all_months = competitor_reference_df['year_month'].unique()
# If your year_month is string, you might want to convert to datetime for sorting:
all_months_date = pd.to_datetime(all_months)
all_months_date = pd.Series(all_months_date).sort_values().unique()


# Build full MultiIndex for own products x all months
full_index_own = pd.MultiIndex.from_product(
    [all_months_date, competitor_skus],
    names=['year_month_date', 'sku']
)

# Convert reference_df_own['year_month'] to datetime for merging
competitor_reference_df['year_month_date'] = pd.to_datetime(competitor_reference_df['year_month'])

# Set index to year_month and sku for easier reindexing
competitor_reference_df = competitor_reference_df.set_index(['year_month_date', 'sku'])

# Reindex to full index
comp_reference_df_own_padded = competitor_reference_df.reindex(full_index_own)

# Fill missing volumes with zero
comp_reference_df_own_padded['reference_volume'] = comp_reference_df_own_padded['reference_volume'].fillna(0)

# Fill missing prices with the mean price per SKU (computed from existing data)
mean_prices = comp_reference_df_own_padded['reference_price'].mean()
comp_reference_df_own_padded['reference_price'] =  comp_reference_df_own_padded['reference_price'].fillna(mean_prices)

# Reset index if you want back to columns
comp_reference_df_own_padded = comp_reference_df_own_padded.reset_index()

year_month_mapping = competitor_reference_df.reset_index()
year_month_mapping = year_month_mapping[['year_month_date', 'year_month']].drop_duplicates()

comp_reference_df_own_padded = pd.merge(comp_reference_df_own_padded, year_month_mapping, how = 'left', on = 'year_month_date')
comp_reference_df_own_padded = comp_reference_df_own_padded.drop(columns = {'year_month_date', 'year_month_x'})
comp_reference_df_own_padded.rename(columns = {'year_month_y': 'year_month'}, inplace = True)



reference_df = reference_df_own_padded.copy()
competitor_reference_df = comp_reference_df_own_padded.copy()

In [ ]:
# Prepare Elasticity Matrix
own_products = reference_df['sku'].unique()
competitor_products = competitor_reference_df['sku'].unique()
all_products = np.concatenate([own_products, competitor_products])

months = sorted(reference_df['year_month'].unique())
num_months = len(months)
num_own = len(own_products)
num_comp = len(competitor_products)

# For convenience, convert these to numpy arrays and keep track of product order
own_index = {p: i for i, p in enumerate(own_products)}
competitor_index = {p: i for i, p in enumerate(competitor_products)}


E_price_to_volume = np.zeros((num_own, num_own))  # shape (price_changer, volume_changer)

for _, row in own_to_own_elasticity_df.iterrows():
    i = own_index[row['target_sku']]  # price changer
    j = own_index[row['other_sku']]   # volume changer
    E_price_to_volume[i, j] = row['elasticity']

E_price_to_comp_volume = np.zeros((num_own, num_comp))  # price changers = own products, volume changers = competitor products

for _, row in own_to_competitor_elasticity_df.iterrows():
    i = own_index[row['target_sku']]  # price changer (own product)
    j = competitor_index[row['other_sku']]  # volume changer (competitor product)
    E_price_to_comp_volume[i, j] = row['elasticity']

In [ ]:
# Size and Segment Extraction
def extract_pack_type(sku):
    known_pack_types = ['NRB', 'RB', 'CAN']
    for pt in known_pack_types:
        if pt in sku:
            return pt
    return 'UNKNOWN'

reference_df['pack_type'] = reference_df['sellin_sku'].apply(extract_pack_type)
reference_df.loc[(reference_df['pack_type']=='UNKNOWN') & (reference_df['sku'].str.contains('NO RETORNABLE')), 'pack_type'] = 'NRB'
reference_df.loc[(reference_df['pack_type']=='UNKNOWN') & (reference_df['sku'].str.contains('RETORNABLE')), 'pack_type'] = 'RB'

def assign_size_group(row):
    cap = row['capacity']
    pt = row['pack_type']
    
    if cap < 300:
        return 'Small'
    elif pt == 'CAN':
        if 300 <= cap <= 399:
            return 'Regular'
        elif cap > 399:
            return 'Large'
    elif pt in ['RB', 'NRB']:
        if 300 <= cap <= 599:
            return 'Regular'
        elif cap > 599:
            return 'Large'
    return 'Unknown'

# Apply to dataframe
reference_df['size_group'] = reference_df.apply(assign_size_group, axis=1)

In [ ]:
# Helper function to get reference arrays per month
def get_reference_arrays_own(month, ref_df, product_list):
    df_m = ref_df[(ref_df['year_month'] == month) & (ref_df['sku'].isin(product_list))].set_index('sku').reindex(product_list)
    # Fill missing with zeros or appropriate defaults
    #df_m = df_m.fillna(0)
    volume = df_m['reference_volume'].values
    volume_sellout = df_m['sellout_volume'].values
    price = df_m['reference_price'].values
    capacity = df_m['capacity'].values
    markup = df_m['markup'].values
    discount = df_m['discount'].values
    excise = df_m['excise'].values
    vilc = df_m['vilc'].values
    return volume, volume_sellout, price, capacity, markup, discount, excise, vilc


def get_reference_arrays_comp(month, ref_df, product_list):
    df_m = ref_df[(ref_df['year_month'] == month) & (ref_df['sku'].isin(product_list))].set_index('sku').reindex(product_list)
    # Fill missing with zeros or appropriate defaults
    #df_m = df_m.fillna(0)
    volume = df_m['reference_volume'].values
    price = df_m['reference_price'].values
    return volume, price

In [ ]:
# Volume calculation given price changes and elasticities
def calc_volume(opt_price_unit, ref_volume_own, ref_volume_sellout, ref_volume_comp, ref_price_liter, capacity, elasticity_matrix_own, elasticity_matrix_comp):
    opt_price_liter = opt_price_unit * 1000 / capacity
    price_ratio_own = np.log(opt_price_liter / ref_price_liter)

    Q_own_opt = ref_volume_own * np.exp(elasticity_matrix_own.T @ price_ratio_own)
    Q_own_opt_sellout = ref_volume_sellout * np.exp(elasticity_matrix_own.T @ price_ratio_own)
    Q_comp_opt = ref_volume_comp * np.exp(elasticity_matrix_comp.T @ price_ratio_own)

    return Q_own_opt, Q_comp_opt, Q_own_opt_sellout

In [ ]:
# MACO and NR Calculation based on volume and costs
def calc_MACO(opt_price_unit, opt_volume, capacity, markup, discount, excise, vilc):
    sales_units = (opt_volume * 100000) / capacity
    discount_pct = discount 
    excise_pct = excise      
    NR_per_unit = ((opt_price_unit / (1 + markup)) * (1 + discount_pct + excise_pct)) / (1 + VAT)
    NR = NR_per_unit * sales_units
    MACO = NR + vilc
    return NR, MACO, sales_units

In [ ]:
# FUNCTION TO CALCULATE OWN AND COMPETITOR OPTIMIZED VOLUME - SELLOUT
def get_industry_volumes(P_opt):

    prices_by_month = P_opt.reshape(num_months, num_own)

    abi_sku_all = []
    abi_year_month_all = []
    # abi_price_liter_ref_all = []
    # abi_price_unit_ref_all = []
    abi_volume_ref_all = []
    abi_volume_opt_all = []
    abi_volume_ref_sellout_all = []
    abi_volume_opt_sellout_all = []

    comp_sku_all = []
    comp_year_month_all = []
    # comp_price_liter_ref_all = []
    # comp_price_unit_ref_all = []
    comp_volume_ref_all = []
    comp_volume_opt_all = []

    for i, month in enumerate(months):
        # Get reference arrays for own products and competitors for this month
        (ref_vol_own, ref_vol_sellout, ref_price_own, capacity_own,
         markup_own, discount_own, excise_own, vilc_own) = get_reference_arrays_own(month, reference_df, own_products)

        (ref_vol_comp, ref_price_comp) = get_reference_arrays_comp(month, competitor_reference_df, competitor_products)

        price_opt_own = prices_by_month[i, :]
        price_opt_liter = price_opt_own * 1000 / capacity_own
        price_ref_unit = ref_price_own * capacity_own / 1000

        # Calculate own volumes
        vol_own_opt, vol_comp_opt, vol_own_opt_sellout = calc_volume(price_opt_own, ref_vol_own, ref_vol_sellout, ref_vol_comp, ref_price_own, 
                                                capacity_own, E_price_to_volume, E_price_to_comp_volume)

        abi_sku_all.extend(own_products)
        abi_year_month_all.extend([month] * len(own_products))
        abi_volume_ref_all.extend(ref_vol_own)
        abi_volume_ref_sellout_all.extend(ref_vol_sellout)
        abi_volume_opt_all.extend(vol_own_opt)
        abi_volume_opt_sellout_all.extend(vol_own_opt_sellout)
        comp_sku_all.extend(competitor_products)
        comp_year_month_all.extend([month] * len(competitor_products))
        comp_volume_ref_all.extend(ref_vol_comp)
        comp_volume_opt_all.extend(vol_comp_opt)
        
        

    global industry_volume_df
    abi_vol_df = pd.DataFrame({
        'sku': abi_sku_all,
        'manufacturer': 'abi',
        'year_month': abi_year_month_all,
        'volume_ref': abi_volume_ref_sellout_all,
        'volume_opt': abi_volume_opt_sellout_all,
    })
    comp_vol_df = pd.DataFrame({
        'sku': comp_sku_all,
        'manufacturer': 'competitor',
        'year_month': comp_year_month_all,
        'volume_ref': comp_volume_ref_all,
        'volume_opt': comp_volume_opt_all,
    })
    industry_volume_df = pd.concat([abi_vol_df, comp_vol_df])
    return industry_volume_df

In [ ]:
# INDUSTRY VOLUME CONSTRAINT - SELLOUT
total_ref_industry_volume = np.sum(reference_df['sellout_volume'].values) + np.sum(competitor_reference_df['reference_volume'].values)

def industry_volume_constraint_monthly(P_opt_unit):
    prices_by_month = P_opt_unit.reshape(num_months, num_own)
    
    total_industry_volume_ref = 0
    total_industry_volume_opt = 0
    total_own_volume_ref = 0
    total_own_volume_opt = 0
    
    for i, month in enumerate(months):

        # Get reference arrays for own products and competitors for this month
        (ref_vol_own, ref_vol_sellout, ref_price_own, capacity_own,
         markup_own, discount_own, excise_own, vilc_own) = get_reference_arrays_own(month, reference_df, own_products)

        (ref_vol_comp, ref_price_comp) = get_reference_arrays_comp(month, competitor_reference_df, competitor_products)

        price_opt_own = prices_by_month[i, :]
        price_opt_liter = price_opt_own * 1000 / capacity_own
        price_ref_unit = ref_price_own * capacity_own / 1000

        # Calculate own volumes
        vol_own_opt, vol_comp_opt, vol_own_opt_sellout = calc_volume(price_opt_own, ref_vol_own, ref_vol_sellout, ref_vol_comp, ref_price_own, 
                                                capacity_own, E_price_to_volume, E_price_to_comp_volume)

        industry_vol_ref = ref_vol_sellout.sum() + ref_vol_comp.sum()
        industry_vol_opt = vol_own_opt_sellout.sum() + vol_comp_opt.sum()

        total_industry_volume_ref += industry_vol_ref
        total_industry_volume_opt += industry_vol_opt
        total_own_volume_ref += ref_vol_sellout.sum()
        total_own_volume_opt += vol_own_opt_sellout.sum()
        
    return total_industry_volume_opt

ind_vol_nlc_monthly = NonlinearConstraint(
    industry_volume_constraint_monthly,
    0.99 * (total_ref_industry_volume * (1- 0.56*target_delta)),
    np.inf
    #1.05 * total_ref_industry_volume
)

In [ ]:
# ABI VOLUME CONSTRAINT - SELLIN
total_ref_own_volume = np.sum(reference_df['reference_volume'].values)

def own_volume_constraint_monthly(P_opt_unit):
    prices_by_month = P_opt_unit.reshape(num_months, num_own)
    
    total_industry_volume_ref = 0
    total_industry_volume_opt = 0
    total_own_volume_ref = 0
    total_own_volume_opt = 0
    
    for i, month in enumerate(months):
        # Get reference arrays for own products and competitors for this month
        (ref_vol_own, ref_vol_sellout, ref_price_own, capacity_own,
         markup_own, discount_own, excise_own, vilc_own) = get_reference_arrays_own(month, reference_df, own_products)
    
        (ref_vol_comp, ref_price_comp) = get_reference_arrays_comp(month, competitor_reference_df, competitor_products)
    
        price_opt_own = prices_by_month[i, :]
    
        vol_own_opt, vol_comp_opt, vol_own_opt_sellout = calc_volume(price_opt_own, ref_vol_own, ref_vol_sellout, ref_vol_comp, ref_price_own, 
                                                capacity_own, E_price_to_volume, E_price_to_comp_volume)
    
        industry_vol_ref = ref_vol_own.sum() + ref_vol_comp.sum()
        industry_vol_opt = vol_own_opt.sum() + vol_comp_opt.sum()
    
        total_industry_volume_ref += industry_vol_ref
        total_industry_volume_opt += industry_vol_opt
        total_own_volume_ref += ref_vol_own.sum()
        total_own_volume_opt += vol_own_opt.sum()
        
    return total_own_volume_opt

own_vol_nlc_monthly = NonlinearConstraint(
    own_volume_constraint_monthly, 
    0.99 * total_ref_own_volume, 
    1.05 * total_ref_own_volume
)

In [ ]:
# MARKET SHARE CONSTRAINT - SELLOUT
ref_market_share = np.sum(reference_df['sellout_volume'].values) / total_ref_industry_volume

def market_share_constraint_monthly(P_opt_unit):
    prices_by_month = P_opt_unit.reshape(num_months, num_own)
    
    total_industry_volume_ref = 0
    total_industry_volume_opt = 0
    total_own_volume_ref = 0
    total_own_volume_opt = 0
    
    for i, month in enumerate(months):
        # Get reference arrays for own products and competitors for this month
        (ref_vol_own, ref_vol_sellout, ref_price_own, capacity_own,
         markup_own, discount_own, excise_own, vilc_own) = get_reference_arrays_own(month, reference_df, own_products)

        (ref_vol_comp, ref_price_comp) = get_reference_arrays_comp(month, competitor_reference_df, competitor_products)

        price_opt_own = prices_by_month[i, :]
        price_opt_liter = price_opt_own * 1000 / capacity_own
        price_ref_unit = ref_price_own * capacity_own / 1000

        # Calculate own volumes
        vol_own_opt, vol_comp_opt, vol_own_opt_sellout = calc_volume(price_opt_own, ref_vol_own, ref_vol_sellout, ref_vol_comp, ref_price_own, 
                                                capacity_own, E_price_to_volume, E_price_to_comp_volume)

        industry_vol_ref = ref_vol_sellout.sum() + ref_vol_comp.sum()
        industry_vol_opt = vol_own_opt_sellout.sum() + vol_comp_opt.sum()

        total_industry_volume_ref += industry_vol_ref
        total_industry_volume_opt += industry_vol_opt
        total_own_volume_ref += ref_vol_sellout.sum()
        total_own_volume_opt += vol_own_opt_sellout.sum()
        #print(total_industry_volume_ref)
        #print(total_own_volume_opt)
        #print(vol_own_opt_sellout)
        
    ms_opt = total_own_volume_opt / total_industry_volume_opt
    
    return ms_opt

market_share_nlc_monthly = NonlinearConstraint(
    market_share_constraint_monthly, 
    ref_market_share - 0.005, np.inf 
)

In [ ]:
# PINC CONSTRAINT - SELLIN
tolerance = 0.005
weighted_avg_ref_price = np.sum(reference_df['reference_volume'].values * reference_df['reference_price'].values) / np.sum(reference_df['reference_volume'].values)

def portfolio_pinc_constraint_monthly(P_opt_unit):
    prices_by_month = P_opt_unit.reshape(num_months, num_own)
    
    total_industry_volume_ref = 0
    total_industry_volume_opt = 0
    total_own_volume_ref = 0
    total_own_volume_opt = 0
    total_price_mult_vol_opt = 0
    
    for i, month in enumerate(months):
        # Get reference arrays for own products and competitors for this month
        (ref_vol_own, ref_vol_sellout, ref_price_own, capacity_own,
         markup_own, discount_own, excise_own, vilc_own) = get_reference_arrays_own(month, reference_df, own_products)
    
        (ref_vol_comp, ref_price_comp) = get_reference_arrays_comp(month, competitor_reference_df, competitor_products)
    
        price_opt_own = prices_by_month[i, :]
        price_opt_own_liter = price_opt_own * 100000 / capacity_own
    
        vol_own_opt, vol_comp_opt, vol_own_opt_sellout = calc_volume(price_opt_own, ref_vol_own, ref_vol_sellout, ref_vol_comp, ref_price_own, 
                                                capacity_own, E_price_to_volume, E_price_to_comp_volume)

        price_mult_vol_opt = np.sum(price_opt_own_liter * vol_own_opt)
    
        industry_vol_ref = ref_vol_own.sum() + ref_vol_comp.sum()
        industry_vol_opt = vol_own_opt.sum() + vol_comp_opt.sum()
    
        total_industry_volume_ref += industry_vol_ref
        total_industry_volume_opt += industry_vol_opt
        total_own_volume_ref += ref_vol_own.sum()
        total_own_volume_opt += vol_own_opt.sum()
        total_price_mult_vol_opt += price_mult_vol_opt
        
    weighted_avg_opt_price = total_price_mult_vol_opt / (total_own_volume_opt * 100)
    
    return weighted_avg_opt_price

portfolio_pinc_nlc_monthly = NonlinearConstraint(
    portfolio_pinc_constraint_monthly, 
    ((1 + target_delta) * weighted_avg_ref_price) - tolerance,
    ((1 + target_delta) * weighted_avg_ref_price) + tolerance
)

In [ ]:
# SOFT CONSTRAINT - MULTIPLE OF 50
def multiples_of_50_penalty(P_opt, P_ref, penalty_weight=1.0):
    # P_opt, P_ref are numpy arrays of optimized and reference per unit prices
    diff = P_opt - P_ref
    remainder = np.mod(diff, 50)
    penalty_per_sku = np.minimum(remainder**2, (50 - remainder)**2)
    total_penalty = penalty_weight * np.sum(penalty_per_sku)
    return total_penalty

def round_to_nearest_50(price_array, ref_price_array):
    # price_array and ref_price_array are same shape arrays of optimized and ref prices
    price_diff = price_array - ref_price_array
    price_diff_rounded = 50 * np.round(price_diff / 50)
    return ref_price_array + price_diff_rounded

In [ ]:
# DEFINING BOUNDS
reference_df_padded_bound['reference_price_unit'] = (
    reference_df_padded_bound['reference_price'] * reference_df_padded_bound['capacity'] / 1000
)


# Flatten the reference prices (matching the order of your optimization vector)
ref_prices = reference_df_padded_bound['reference_price_unit'].values

# Create a boolean mask: True if SKU has actual data, False if it's padded
# For this example, assume that padded SKUs have volume = 0 or reference_price = NaN originally
# Adjust condition as per your data
is_present = reference_df_padded_bound['present'] == 'present'  # True for present, False for padded

# If some padded rows have NaN reference_price, fill those NaNs temporarily for bounds calc
ref_prices_filled = np.where(np.isnan(ref_prices), 0, ref_prices)

bounds = []
for present, ref_price in zip(is_present, ref_prices_filled):
    if present:
        lower = max(ref_price - 300, 0)  # Prices can't be negative
        upper = ref_price + 500
        bounds.append((lower, upper))
    else:
        # Fixed price for missing SKU: bounds are equal so optimizer won't change it
        #print('i entered here')
        bounds.append((ref_price, ref_price))

# bounds is now a list of (lower, upper) tuples for each sku x month in correct order


In [ ]:
# OBJECTIVE FUNCTION
def objective(P_opt):

    total_MACO = 0
    penalty = 0

    # Reshape to [num_months, num_own]
    prices_by_month = P_opt.reshape(num_months, num_own)

    # Initialize accumulators for constraints etc
    total_industry_volume_ref = 0
    total_industry_volume_opt = 0
    total_own_volume_ref = 0
    total_own_volume_opt = 0
    total_own_revenue_ref = 0
    total_own_revenue_opt = 0

    # For hierarchy penalties (segment and size group), accumulate per month data
    volumes_by_month = []
    revenues_by_month = []
    sku_all = []
    year_month_all = []
    price_liter_ref_all = []
    price_unit_ref_all = []
    volume_ref_all = []
    volume_ref_sellout_all = []
    NR_ref_all = []
    MACO_ref_all = []
    price_liter_opt_all = []
    price_unit_opt_all = []
    volume_opt_all = []
    volume_opt_sellout_all = []
    NR_opt_all = []
    MACO_opt_all = []

    for i, month in enumerate(months):
        # Get reference arrays for own products and competitors for this month
        (ref_vol_own, ref_vol_sellout, ref_price_own, capacity_own,
         markup_own, discount_own, excise_own, vilc_own) = get_reference_arrays_own(month, reference_df, own_products)
    
        (ref_vol_comp, ref_price_comp) = get_reference_arrays_comp(month, competitor_reference_df, competitor_products)

        price_opt_own = prices_by_month[i, :]
        price_opt_liter = price_opt_own * 1000 / capacity_own
        price_ref_unit = ref_price_own * capacity_own / 1000

        # Calculate own volumes
        vol_own_opt, vol_comp_opt, vol_own_opt_sellout = calc_volume(price_opt_own, ref_vol_own, ref_vol_sellout, ref_vol_comp, ref_price_own, 
                                                capacity_own, E_price_to_volume, E_price_to_comp_volume)


        # Aggregate industry volumes for this month
        industry_vol_ref = ref_vol_own.sum() + ref_vol_comp.sum()
        industry_vol_opt = vol_own_opt.sum() + vol_comp_opt.sum()

        total_industry_volume_ref += industry_vol_ref
        total_industry_volume_opt += industry_vol_opt
        total_own_volume_ref += ref_vol_own.sum()
        total_own_volume_opt += vol_own_opt.sum()

        # Calculate NR and MACO for own products for this month
        NR_own, MACO_own, _ = calc_MACO(price_opt_own, vol_own_opt, capacity_own,
                                       markup_own, discount_own, excise_own, vilc_own)

        NR_ref, MACO_ref, _ = calc_MACO(price_ref_unit, ref_vol_own, capacity_own,
                                       markup_own, discount_own, excise_own, vilc_own)

        total_own_revenue_opt += NR_own.sum()
        total_MACO += MACO_own.sum()

        sku_all.extend(own_products)
        #year_month_all.extend(month)
        year_month_all.extend([month] * len(own_products))
        price_liter_ref_all.extend(ref_price_own)
        price_unit_ref_all.extend(price_ref_unit)
        volume_ref_all.extend(ref_vol_own)
        volume_ref_sellout_all.extend(ref_vol_sellout)
        NR_ref_all.extend(NR_ref)
        MACO_ref_all.extend(MACO_ref)
        price_liter_opt_all.extend(price_opt_liter)
        price_unit_opt_all.extend(price_opt_own)
        volume_opt_all.extend(vol_own_opt)
        volume_opt_sellout_all.extend(vol_own_opt_sellout)
        NR_opt_all.extend(NR_own)
        MACO_opt_all.extend(MACO_own)
        

    global monthly_outputs_df
    monthly_outputs_df = pd.DataFrame({
        'sku': sku_all,
        'year_month': year_month_all,
        'price_liter_ref': price_liter_ref_all,
        'price_unit_ref': price_unit_ref_all,
        'volume_ref': volume_ref_all,
        'volume_ref_sellout': volume_ref_sellout_all,
        'NR_ref': NR_ref_all,
        'MACO_ref': MACO_ref_all,
        'price_liter_opt': price_liter_opt_all,
        'price_unit_opt': price_unit_opt_all,
        'volume_opt': volume_opt_all,
        'volume_opt_sellout': volume_opt_sellout_all,
        'NR_opt': NR_opt_all,
        'MACO_opt': MACO_opt_all
    })

    monthly_outputs_df = monthly_outputs_df.merge(
        reference_df[['sku', 'year_month', 'segment', 'size_group', 'capacity']],
        on=['sku', 'year_month'],
        how='left'
    )
    monthly_outputs_df['HL'] = monthly_outputs_df['volume_opt']

    # Total NR and HL per segment
    segment_group = monthly_outputs_df.groupby('segment').agg({
        'NR_opt': 'sum',
        'volume_opt': 'sum'
    })
    segment_group['NR_per_HL'] = segment_group['NR_opt'] / segment_group['volume_opt']

    # Total NR and HL per size group
    size_group = monthly_outputs_df.groupby('size_group').agg({
        'NR_opt': 'sum',
        'volume_opt': 'sum'
    })
    size_group['NR_per_HL'] = size_group['NR_opt'] / size_group['volume_opt']

    segment_order = ['Value', 'Core', 'Core+', 'Premium', 'Super Premium']
    size_order = ['Small', 'Regular', 'Large']

    segment_penalty = 0 
    size_penalty = 0 

    # Segment NR/HL hierarchy penalty
    for i in range(len(segment_order) - 1):
        seg_a = segment_order[i]
        seg_b = segment_order[i + 1]
        if seg_a in segment_group.index and seg_b in segment_group.index:
            diff = segment_group.loc[seg_a, 'NR_per_HL'] - segment_group.loc[seg_b, 'NR_per_HL']
            if diff > 0:
                segment_penalty += diff * 1e5  
    
    # Size group NR/HL hierarchy penalty (Small > Regular > Large)
    for i in range(len(size_order) - 1):
        sz_a = size_order[i]
        sz_b = size_order[i + 1]
        if sz_a in size_group.index and sz_b in size_group.index:
            diff = size_group.loc[sz_b, 'NR_per_HL'] - size_group.loc[sz_a, 'NR_per_HL']
            if diff > 0:
                size_penalty += diff * 1e5


    mult50_penalty_weight = 1e5
    mult50_penalty = multiples_of_50_penalty(P_opt, reference_df['reference_price'] * reference_df['capacity'] / 1000, mult50_penalty_weight)

    #print('Total MACO: ', total_MACO)
    #print('Segment Penalty: ', segment_penalty)
    #print('Size Penalty: ', size_penalty)
    #print(monthly_outputs_df)
    #monthly_outputs_df.to_csv(r"Round 2/Initial Trial Input/Monthly Trial/output df intermediate from objective.csv", index=False)
    # Objective: maximize MACO -> minimize negative MACO + penalty
    return -total_MACO + segment_penalty + size_penalty + mult50_penalty

In [ ]:
import time

reference_df_ordered = reference_df.set_index(['sku', 'year_month']).loc[
    [(sku, month) for month in months for sku in own_products]
].reset_index()

reference_df_ordered['reference_price_unit'] = (
    reference_df_ordered['reference_price'] * reference_df_ordered['capacity'] / 1000
)

# Setup initial guess for prices (flattened over months)
P0 = reference_df_ordered['reference_price_unit'].values

# === Run optimization ===
start = time.time()
result = minimize(objective, 
                  P0,
                  method='trust-constr',
                  bounds=bounds, 
                  constraints=[own_vol_nlc_monthly, ind_vol_nlc_monthly, market_share_nlc_monthly, portfolio_pinc_nlc_monthly],
                  #constraints=[own_vol_nlc, ind_vol_nlc, market_share_nlc],
                  options={'disp': True})

print(f"Optimization took {time.time() - start:.4f} seconds")
opt_industry_volume_df = get_industry_volumes(result.x)
monthly_outputs_df_unrounded = monthly_outputs_df.copy()

In [ ]:
P0

In [ ]:
# CHECKING CONSTRAINTS ADHERENCE

# Calculate total volumes
total_ref_volume = opt_industry_volume_df['volume_ref'].sum()
total_opt_volume = opt_industry_volume_df['volume_opt'].sum()
abi_ref_volume = opt_industry_volume_df[opt_industry_volume_df['manufacturer']=='abi']['volume_ref'].sum()
abi_opt_volume = opt_industry_volume_df[opt_industry_volume_df['manufacturer']=='abi']['volume_opt'].sum()
comp_ref_volume = opt_industry_volume_df[opt_industry_volume_df['manufacturer']=='competitor']['volume_ref'].sum()
comp_opt_volume = opt_industry_volume_df[opt_industry_volume_df['manufacturer']=='competitor']['volume_opt'].sum()


print("TOTAL MACO: ")
total_maco_reference = monthly_outputs_df['MACO_ref'].sum()
total_maco_optimized = monthly_outputs_df['MACO_opt'].sum()
print(f"Reference MACO: {total_maco_reference:,.0f}")
print(f"Optimized MACO: {total_maco_optimized:,.0f}")
print(f"MACO change: {total_maco_optimized/ total_maco_reference - 1:,.4f}")
if total_maco_optimized < total_maco_reference:
    print("❌MACO declined after optimization.")
else:
    print("✅MACO grew after optimization.")


print('\n')
print('INDUSTRY VOLUME CONSTRAINT: ')
print(f"Total industry reference volume: {total_ref_volume:,.2f}")
print(f"Total industry optimized volume: {total_opt_volume:,.2f}")
print(f"Total industry volume change: {total_opt_volume/ total_ref_volume - 1:,.4f}")

lower_bound = 0.99 * (total_ref_volume * (1- 0.56*target_delta))
#upper_bound = 1.05 * total_ref_volume

if total_opt_volume < lower_bound:
    print("❌Constraint violated: total industry volume decreased by more than 1%.")
# elif total_opt_volume > upper_bound:
#     print("❌Constraint violated: total industry volume increased by more than 5%.")
else:
    print("✅Constraint satisfied: total industry volume within allowed bounds.")

print('\n')
print('OWN VOLUME CONSTRAINT: ')
print(f"Total ABI reference volume: {abi_ref_volume:,.2f}")
print(f"Total ABI optimized volume: {abi_opt_volume:,.2f}")
print(f"Total ABI volume change: {abi_opt_volume / abi_ref_volume - 1:,.4f}")

lower_bound = 0.99 * abi_ref_volume
upper_bound = 1.05 * abi_ref_volume

if abi_opt_volume < lower_bound:
    print("❌Constraint violated: ABI volume decreased by more than 1%.")
elif abi_opt_volume > upper_bound:
    print("❌Constraint violated: ABI volume increased by more than 5%.")
else:
    print("✅Constraint satisfied: ABI volume within allowed bounds (±1% to +5%).")


print('\n')
print('MARKET SHARE CONSTRAINT: ')
ref_ms = abi_ref_volume / total_ref_volume
opt_ms = abi_opt_volume / total_opt_volume
print(f"Reference Market Share: {abi_ref_volume / total_ref_volume:,.4f}")
print(f"Optimized Market Share: {abi_opt_volume / total_opt_volume:,.4f}")
print(f"Market Share Change: {opt_ms - ref_ms:,.4f}")

bound = ref_ms - 0.005

if opt_ms < bound:
    print("❌Constraint violated: ABI market share decreased by more than 0.5%.")
else:
    print("✅Constraint satisfied: ABI market share within allowed bounds.")



print('\n')
print('PORTFOLIO PINC CONSTRAINT: ')

ref_ppl = np.sum(monthly_outputs_df['volume_ref'] * monthly_outputs_df['price_liter_ref']) / np.sum(monthly_outputs_df['volume_ref'])
opt_ppl = np.sum(monthly_outputs_df['volume_opt'] * monthly_outputs_df['price_liter_opt']) / np.sum(monthly_outputs_df['volume_opt'])
pinc_delta = (opt_ppl / ref_ppl) - 1

print(f"Reference Portfolio Price/Liter: {ref_ppl:,.4f}")
print(f"Optimized Portfolio Price/Liter: {opt_ppl:,.4f}")
print(f"Specified PINC: {target_delta:,.4f}")
print(f"Optimized PINC: {pinc_delta:,.4f}")

threshold = 0.005

if np.abs(pinc_delta - target_delta) > tolerance:
    print(f"❌Constraint violated: Portfolio PINC greater than {target_delta}")
else:
    print(f"✅Constraint satisfied: Portfolio PINC equal to {target_delta}")



print('\n')
print('NR/HL HIERARCHY CONSTRAINT: ')
# Grouped NR/HL for segments
segment_order = ['Value', 'Core', 'Core+', 'Premium', 'Super Premium']
segment_grouped = monthly_outputs_df.groupby('segment').agg({
    'NR_opt': 'sum',
    'volume_opt': 'sum'
})
segment_grouped['NR_per_HL'] = segment_grouped['NR_opt'] / segment_grouped['volume_opt']
segment_nrh_agg = segment_grouped['NR_per_HL'].reindex(segment_order)

# print("Aggregated NR/HL by Segment (Total NR / Total Volume):")
# print(segment_nrh_agg)

# Grouped NR/HL for size groups
size_order = ['Small', 'Regular', 'Large']
size_grouped = monthly_outputs_df.groupby('size_group').agg({
    'NR_opt': 'sum',
    'volume_opt': 'sum'
})
size_grouped['NR_per_HL'] = size_grouped['NR_opt'] / size_grouped['volume_opt']
size_nrh_agg = size_grouped['NR_per_HL'].reindex(size_order)

# print("\nAggregated NR/HL by Size Group (Total NR / Total Volume):")
# print(size_nrh_agg)


# Checking if reference data satisfies the conditions
violated_segments = []
for i in range(len(segment_order) - 1):
    if segment_nrh_agg[segment_order[i]] > segment_nrh_agg[segment_order[i + 1]]:
        violated_segments.append((segment_order[i], segment_order[i + 1]))

if violated_segments:
    print("❌Segment NR/HL hierarchy violated between:")
    for seg1, seg2 in violated_segments:
        print(f"  {seg1} > {seg2}")
else:
    print("✅Segment NR/HL hierarchy is satisfied")


# Check size group hierarchy (Small > Regular > Large)
violated_sizes = []
for i in range(len(size_order) - 1):
    if size_nrh_agg[size_order[i]] < size_nrh_agg[size_order[i + 1]]:
        violated_sizes.append((size_order[i], size_order[i + 1]))

if violated_sizes:
    print("❌Size group NR/HL hierarchy violated between:")
    for sg1, sg2 in violated_sizes:
        print(f"  {sg1} < {sg2}")
else:
    print("✅Size group NR/HL hierarchy is satisfied")

In [ ]:
rounded_price_opt = round_to_nearest_50(result.x, P0)

objective(rounded_price_opt)

opt_industry_volume_df_rounded = get_industry_volumes(rounded_price_opt)

In [ ]:
# CHECKING CONSTRAINTS ADHERENCE

# Calculate total volumes
total_ref_volume = opt_industry_volume_df_rounded['volume_ref'].sum()
total_opt_volume = opt_industry_volume_df_rounded['volume_opt'].sum()
abi_ref_volume = opt_industry_volume_df_rounded[opt_industry_volume_df_rounded['manufacturer']=='abi']['volume_ref'].sum()
abi_opt_volume = opt_industry_volume_df_rounded[opt_industry_volume_df_rounded['manufacturer']=='abi']['volume_opt'].sum()
comp_ref_volume = opt_industry_volume_df_rounded[opt_industry_volume_df_rounded['manufacturer']=='competitor']['volume_ref'].sum()
comp_opt_volume = opt_industry_volume_df_rounded[opt_industry_volume_df_rounded['manufacturer']=='competitor']['volume_opt'].sum()


print("TOTAL MACO: ")
total_maco_reference = monthly_outputs_df['MACO_ref'].sum()
total_maco_optimized = monthly_outputs_df['MACO_opt'].sum()
print(f"Reference MACO: {total_maco_reference:,.0f}")
print(f"Optimized MACO: {total_maco_optimized:,.0f}")
print(f"MACO change: {total_maco_optimized/ total_maco_reference - 1:,.4f}")
if total_maco_optimized < total_maco_reference:
    print("❌MACO declined after optimization.")
else:
    print("✅MACO grew after optimization.")
    

print('\n')
print('INDUSTRY VOLUME CONSTRAINT: ')
print(f"Total industry reference volume: {total_ref_volume:,.2f}")
print(f"Total industry optimized volume: {total_opt_volume:,.2f}")
print(f"Total industry volume change: {total_opt_volume/ total_ref_volume - 1:,.4f}")

lower_bound = 0.99 * (total_ref_volume * (1- 0.56*target_delta))
#upper_bound = 1.05 * total_ref_volume

if total_opt_volume < lower_bound:
    print("❌Constraint violated: total industry volume decreased by more than 1%.")
# elif total_opt_volume > upper_bound:
#     print("❌Constraint violated: total industry volume increased by more than 5%.")
else:
    print("✅Constraint satisfied: total industry volume within allowed bounds.")

print('\n')
print('OWN VOLUME CONSTRAINT: ')
print(f"Total ABI reference volume: {abi_ref_volume:,.2f}")
print(f"Total ABI optimized volume: {abi_opt_volume:,.2f}")
print(f"Total ABI volume change: {abi_opt_volume / abi_ref_volume - 1:,.4f}")

lower_bound = 0.99 * abi_ref_volume
upper_bound = 1.05 * abi_ref_volume

if abi_opt_volume < lower_bound:
    print("❌Constraint violated: ABI volume decreased by more than 1%.")
elif abi_opt_volume > upper_bound:
    print("❌Constraint violated: ABI volume increased by more than 5%.")
else:
    print("✅Constraint satisfied: ABI volume within allowed bounds (±1% to +5%).")


print('\n')
print('MARKET SHARE CONSTRAINT: ')
ref_ms = abi_ref_volume / total_ref_volume
opt_ms = abi_opt_volume / total_opt_volume
print(f"Reference Market Share: {abi_ref_volume / total_ref_volume:,.4f}")
print(f"Optimized Market Share: {abi_opt_volume / total_opt_volume:,.4f}")
print(f"Market Share Change: {opt_ms - ref_ms:,.4f}")

bound = ref_ms - 0.005

if opt_ms < bound:
    print("❌Constraint violated: ABI market share decreased by more than 0.5%.")
else:
    print("✅Constraint satisfied: ABI market share within allowed bounds.")



print('\n')
print('PORTFOLIO PINC CONSTRAINT: ')

ref_ppl = np.sum(monthly_outputs_df['volume_ref'] * monthly_outputs_df['price_liter_ref']) / np.sum(monthly_outputs_df['volume_ref'])
opt_ppl = np.sum(monthly_outputs_df['volume_opt'] * monthly_outputs_df['price_liter_opt']) / np.sum(monthly_outputs_df['volume_opt'])
pinc_delta = (opt_ppl / ref_ppl) - 1

print(f"Reference Portfolio Price/Liter: {ref_ppl:,.4f}")
print(f"Optimized Portfolio Price/Liter: {opt_ppl:,.4f}")
print(f"Specified PINC: {target_delta:,.4f}")
print(f"Optimized PINC: {pinc_delta:,.4f}")

threshold = 0.005

if np.abs(pinc_delta - target_delta) > tolerance:
    print(f"❌Constraint violated: Portfolio PINC greater than {target_delta}")
else:
    print(f"✅Constraint satisfied: Portfolio PINC equal to {target_delta}")



print('\n')
print('NR/HL HIERARCHY CONSTRAINT: ')
# Grouped NR/HL for segments
segment_order = ['Value', 'Core', 'Core+', 'Premium', 'Super Premium']
segment_grouped = monthly_outputs_df.groupby('segment').agg({
    'NR_opt': 'sum',
    'volume_opt': 'sum'
})
segment_grouped['NR_per_HL'] = segment_grouped['NR_opt'] / segment_grouped['volume_opt']
segment_nrh_agg = segment_grouped['NR_per_HL'].reindex(segment_order)

# print("Aggregated NR/HL by Segment (Total NR / Total Volume):")
# print(segment_nrh_agg)

# Grouped NR/HL for size groups
size_order = ['Small', 'Regular', 'Large']
size_grouped = monthly_outputs_df.groupby('size_group').agg({
    'NR_opt': 'sum',
    'volume_opt': 'sum'
})
size_grouped['NR_per_HL'] = size_grouped['NR_opt'] / size_grouped['volume_opt']
size_nrh_agg = size_grouped['NR_per_HL'].reindex(size_order)

# print("\nAggregated NR/HL by Size Group (Total NR / Total Volume):")
# print(size_nrh_agg)


# Checking if reference data satisfies the conditions
violated_segments = []
for i in range(len(segment_order) - 1):
    if segment_nrh_agg[segment_order[i]] > segment_nrh_agg[segment_order[i + 1]]:
        violated_segments.append((segment_order[i], segment_order[i + 1]))

if violated_segments:
    print("❌Segment NR/HL hierarchy violated between:")
    for seg1, seg2 in violated_segments:
        print(f"  {seg1} > {seg2}")
else:
    print("✅Segment NR/HL hierarchy is satisfied")


# Check size group hierarchy (Small > Regular > Large)
violated_sizes = []
for i in range(len(size_order) - 1):
    if size_nrh_agg[size_order[i]] < size_nrh_agg[size_order[i + 1]]:
        violated_sizes.append((size_order[i], size_order[i + 1]))

if violated_sizes:
    print("❌Size group NR/HL hierarchy violated between:")
    for sg1, sg2 in violated_sizes:
        print(f"  {sg1} < {sg2}")
else:
    print("✅Size group NR/HL hierarchy is satisfied")

In [ ]:
monthly_outputs_df.to_excel(r'Round 2/Final Input Output/Monthly Run Output 3 Month Subset FIX 50Mult Rounded.xlsx', index=False)

In [ ]:
opt_industry_volume_df_rounded.to_excel(r'Round 2/Final Input Output/Industry Vol Output 3 Month Subset FIX 50Mult Rounded.xlsx', index=False)

# END